### 3.sklearn代码实践

In [ ]:
# --- PCA（主成分分析）降维实践 ---
# PCA 核心思想：通过正交变换将数据投影到方差最大的方向上（主成分）
# 数学原理：对协方差矩阵进行特征值分解，特征值大的方向即为主成分方向
# 目标：在尽可能保留数据信息（方差）的前提下降低维度
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
%matplotlib inline
from sklearn.datasets import make_blobs
# 生成三维聚类数据：10000个样本，3个特征，4个簇
X, y = make_blobs(n_samples=10000, n_features=3, centers=[[3,3, 3], [0,0,0], [1,1,1], [2,2,2]], cluster_std=[0.2, 0.1, 0.2, 0.2], random_state =9)
fig = plt.figure()
ax = Axes3D(fig, rect=[0, 0, 1, 1], elev=30, azim=20)
plt.scatter(X[:, 0], X[:, 1], X[:, 2],marker='o')  # 三维散点图可视化

In [ ]:
# 保留全部3个主成分，分析各主成分的方差贡献率
# explained_variance_ratio_：每个主成分解释的方差占总方差的比例
# explained_variance_：每个主成分方向上的方差绝对值
# PCA 的数学步骤：1) 数据中心化 2) 计算协方差矩阵 3) 特征值分解 4) 选择前k个特征向量
from sklearn.decomposition import PCA
pca = PCA(n_components=3)
pca.fit(X)
print(pca.explained_variance_ratio_)  # [0.983, 0.009, 0.008] 第一主成分占98.3%
print(pca.explained_variance_)

可以看出投影后三个特征维度的方差比例大约为98.3%：0.8%：0.8%。投影后第一个特征占了绝大多数的主成分比例。

In [ ]:
# 从3维降到2维：保留方差最大的2个主成分
# 由于第一主成分占比高达98.3%，降维后信息损失很小
pca = PCA(n_components=2)
pca.fit(X)
print(pca.explained_variance_ratio_)
print(pca.explained_variance_)

这个结果其实可以预料，因为上面三个投影后的特征维度的方差分别为：[ 3.78483785 0.03272285 0.03201892]，投影到二维后选择的肯定是前两个特征，而抛弃第三个特征。

In [ ]:
# 可视化降维后的二维数据
# transform 将原始数据投影到主成分空间：X_new = X * W（W为前k个特征向量组成的矩阵）
X_new = pca.transform(X)
plt.scatter(X_new[:, 0], X_new[:, 1],marker='o')
plt.show()

可见降维后的数据依然可以很清楚的看到我们之前三维图中的4个簇。

现在我们看看不直接指定降维的维度，而指定降维后的主成分方差和比例。

In [ ]:
# 方法一：通过指定方差保留比例自动选择维度
# n_components=0.95 表示保留至少95%方差所需的最少主成分数
# 由于第一主成分已占98.3% > 95%，所以只需1个主成分
pca = PCA(n_components=0.95)
pca.fit(X)
print(pca.explained_variance_ratio_)
print(pca.explained_variance_)
print(pca.n_components_)  # 自动选择保留1个主成分

In [ ]:
# 提高方差保留阈值到99%
# 第一主成分(98.3%)不够，需要加上第二主成分(0.8%)，共约99.1% > 99%
pca = PCA(n_components=0.99)
pca.fit(X)
print(pca.explained_variance_ratio_)
print(pca.explained_variance_)
print(pca.n_components_)  # 自动选择保留2个主成分

这个结果也很好理解，因为我们第一个主成分占了98.3%的方差比例，第二个主成分占了0.8%的方差比例，两者一起可以满足我们的阈值。

最后我们看看让MLE算法自己选择降维维度的效果，代码如下：

In [ ]:
# 方法二：使用 MLE（最大似然估计）算法自动选择最优维度
# MLE 方法通过统计估计确定保留多少主成分最合适
# 适用于数据量充足的情况，结果通常比固定阈值更可靠
pca = PCA(n_components='mle')
pca.fit(X)
print(pca.explained_variance_ratio_)
print(pca.explained_variance_)
print(pca.n_components_)  # MLE 自动选择保留1个主成分

可见由于我们的数据的第一个投影特征的方差占比高达98.3%，MLE算法只保留了我们的第一个特征。